
feature engineering for BodyFat.csv (extended dataset)

adds a 'NavyBodyFat' column using the US Navy circumference method,
and drops the 'Original' y/n column.

US Navy formula (imperial units: inches):
    Men:   %BF = 86.010 * log10(Abdomen - Neck) - 70.041 * log10(Height) + 36.76
    Women: %BF = 163.205 * log10(Waist + Hip - Neck) - 97.684 * log10(Height) - 78.387


In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("BodyFat.csv")

df = df.drop(columns="Original")

df["Height"] = df["Height"] *100 # m to cm
df["Height_in"] = df["Height"] / 2.54
df["Abdomen_in"] = df["Abdomen"] / 2.54
df["Neck_in"] = df["Neck"] / 2.54
df["Hip_in"] = df["Hip"] / 2.54 


def navy_bodyfat_male(abdomen, neck, height):
    return 86.010 * np.log10(abdomen - neck) - 70.041 * np.log10(height) + 36.76


def navy_bodyfat_female(waist, hip, neck, height):
    return 163.205 * np.log10(waist + hip - neck) - 97.684 * np.log10(height) - 78.387


def compute_navy(row):
    is_male = str(row["Sex"]).strip().upper()=="M"
    if is_male:
        return navy_bodyfat_male(row["Abdomen_in"], row["Neck_in"], row["Height_in"])
    else:
        return navy_bodyfat_female(row["Abdomen_in"], row["Hip_in"], row["Neck_in"], row["Height_in"])



df["NavyBodyFat"] = df.apply(compute_navy, axis=1)
df=df.drop(columns=["Hip_in","Neck_in","Abdomen_in","Height_in"])

df.to_csv("BodyFat_engineered.csv", index=False)


Feature usefulness check for BodyFat_engineered.csv

1. Correlation of every numeric feature with BodyFat (target) - all rows
2. Same, split by Sex (M / F) separately

This is a quick screening step - high correlation = likely useful,
low correlation overall but high within one sex = sex-specific signal.


In [8]:
import pandas as pd
df = pd.read_csv("BodyFat_engineered.csv")

EXCLUDE = ["BodyFat", "Density", "NavyBodyFat", "Sex"]

feature_cols = [c for c in df.columns if c not in EXCLUDE and pd.api.types.is_numeric_dtype(df[c])]


def correlation_report(data: pd.DataFrame):
    
    corrs = data[feature_cols].corrwith(data["BodyFat"]).sort_values(key=abs, ascending=False)
    for feat, val in corrs.items():
        print(f"  {feat}         {val:+.3f}")

print("all rows")
correlation_report(df)




all rows
  Hip         +0.588
  Knee         +0.367
  Abdomen         +0.357
  Weight         +0.346
  Thigh         +0.331
  Chest         +0.310
  Biceps         +0.258
  Ankle         +0.183
  Height         -0.151
  Neck         +0.122
  Forearm         +0.106
  Wrist         +0.089
  Age         +0.016


In [7]:
print("male")
male = df[df["Sex"].str.strip().str.upper() == "M"]
correlation_report(male)

print("female")
female = df[df["Sex"].str.strip().str.upper() == "F"]
correlation_report(female)

male
  Abdomen         +0.813
  Chest         +0.703
  Hip         +0.625
  Weight         +0.612
  Thigh         +0.560
  Knee         +0.509
  Biceps         +0.493
  Neck         +0.491
  Forearm         +0.361
  Wrist         +0.347
  Age         +0.291
  Ankle         +0.266
  Height         -0.090
female
  Abdomen         +0.740
  Hip         +0.715
  Weight         +0.701
  Biceps         +0.670
  Thigh         +0.633
  Knee         +0.628
  Forearm         +0.586
  Neck         +0.498
  Wrist         +0.497
  Chest         +0.438
  Ankle         +0.431
  Age         -0.058
  Height         +0.013


The "all rows" correlations are misleadingly weak compared to per-sex. Abdomen jumps from 0.357 (all) to 0.81/0.74 split.

This alone is strong evidence we should train separate models per sex.


1. Age is the standout sex-difference feature. Positive for males (+0.29, older men tend toward more fat) but essentially zero/slightly negative for females (-0.06).
2. Forearm, Biceps, Wrist, Ankle, Knee are all more predictive for females than males.
3. Height is essentially useless in both groups (-0.09, +0.01).
4. Chest is much more informative for men (0.70) than women (0.44).